# Guide 9: Alignment: teaching a model which answer is better

*Lifecycle* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/alignment/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/09-alignment.ipynb)

This notebook holds every code block from Guide 9, in the guide's order: 8 code cells, 7 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- Why "better" needs a new kind of data — no code
- The toy preference task — 1 code cell
- Reward models — 2 code cells
- Optimizing against a reward model, and reward hacking — 2 code cells
- Direct preference optimization — 2 code cells
- When the score lies: over-optimization — 1 code cell
- Variations, and what alignment doesn't do — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## The toy preference task

The code uses PyTorch 2.14.1 from the series `requirements.txt`, as of October 2026, and its blocks run in order and share variables; run them from the repository root, so the first block finds `guides/`. The plumbing (prompts, padding, training loops) lives in `guides/preference.py`, in the repository; the blocks show the lines that carry each lesson. A response's probability is the product of the model's next-token probabilities along it (in the toy, each character is a token). Writing $x$ for the prompt (unrelated to the $x$ of Guides 1 and 6) and $\pi$ for the model's probability of a whole response, $\pi(\text{"rug"} \mid x) = p(\text{r}) \cdot p(\text{u} \mid \text{r}) \cdot p(\text{g} \mid \text{ru})$, each given $x$ too, so its log-probability is the sum of their logs. `place_dist` computes that for each of the six places after a prompt, then renormalizes over the six. `home_rate` averages the probability of each animal's own home.

In [2]:
import sys
sys.path.insert(0, "guides")                   # run from the repository root
import torch
import torch.nn.functional as F
from preference import (load_base, prompts, places, trained, rest, place_dist, home_rate, home_index,
                        pairs_for, train_rm, rm_accuracy, rm_scores, draws, kl_to_ref,
                        dpo, train_prompts, test_prompts, breakdown, val_loss, generate)

base = load_base()                             # Guide 8's base model, from its checkpoint
print(f"home: cat, dog, rat {home_rate(base, prompts(trained)):.1%}; fox, owl, hen {home_rate(base, prompts(rest)):.1%}")
print({y: round(p, 3) for y, p in zip(places, place_dist(base, ["the fox lay under the "])[0].tolist())})

home: cat, dog, rat 18.9%; fox, owl, hen 17.1%
{'mat': 0.244, 'log': 0.163, 'rug': 0.088, 'bed': 0.165, 'pot': 0.235, 'hut': 0.106}


## Reward models

Here $\succ$ reads "is preferred to". Only the difference matters. The loss is the negative log of this probability, the same cross-entropy as Guide 1, the negative log of the probability given to the correct answer, applied to a two-way choice: it's small when the preferred response scores well above the rejected one.

In [3]:
def bt_loss(r_pref, r_rej):                    # −log sigmoid(R(x, y+) − R(x, y−)), averaged over pairs
    return -F.logsigmoid(r_pref - r_rej).mean()

print(f"{torch.sigmoid(torch.tensor(2.0 - 0.5)):.3f} {torch.sigmoid(torch.tensor(0.0)):.3f}")

0.818 0.500


Now train a reward model on pairs for three animals only, the cat, dog and rat: 48 prompts, 240 pairs, 200 steps of 32 pairs with Adam at 0.001. `train_rm` uses the `bt_loss` just defined. Then check how often it ranks the home above another place, for the animals it trained on and for the three it didn't.

In [4]:
rm = train_rm(pairs_for(prompts(trained)), bt_loss, steps=200, seed=0)   # about 5 s
print(f"pairs ranked right: cat, dog, rat {rm_accuracy(rm, prompts(trained)):.1%}; fox, owl, hen {rm_accuracy(rm, prompts(rest)):.1%}")
scores = rm_scores(rm, [f"the {a} sat on the " for a in rest])          # 3 prompts × 6 places
print({a: [places[i] for i in s.argsort(descending=True)[:3]] for a, s in zip(rest, scores)})

pairs ranked right: cat, dog, rat 100.0%; fox, owl, hen 21.2%
{'fox': ['bed', 'log', 'hut'], 'owl': ['bed', 'hut', 'log'], 'hen': ['bed', 'hut', 'log']}


## Optimizing against a reward model, and reward hacking

Real RLHF can't list every possible response, so it samples and updates with PPO. With only six places per prompt the toy can list them, so best-of-n here and the exact optimum below stand in for step 3. The simplest optimizer, best-of-n, has no guard at all. **Best-of-n** samples $n$ responses (here $n$ counts samples, not sequence length) and keeps the one the reward model scores highest. The block samples places from the base model, 12 times per prompt for each $n$ (576 selections per group), and counts how often the kept place is the animal's real home.

In [5]:
for n in (1, 2, 4, 8, 16):
    row = []
    for group in (trained, rest):
        batch = prompts(group)                                      # 48 prompts
        scores, home_idx = rm_scores(rm, batch), home_index(batch)  # 48 × 6 scores; each prompt's home
        hits = [samples[scores[k][samples].argmax()] == home_idx[k]     # keep the highest-scoring sample; is it the home?
                for k, sets in enumerate(draws(base, batch, n)) for samples in sets]   # 12 sets of n places per prompt
        row.append(sum(hits).item() / len(hits))
    print(f"n={n:2}  home chosen: cat, dog, rat {row[0]:.1%}; fox, owl, hen {row[1]:.1%}")

n= 1  home chosen: cat, dog, rat 22.4%; fox, owl, hen 14.2%


n= 2  home chosen: cat, dog, rat 33.2%; fox, owl, hen 11.1%


n= 4  home chosen: cat, dog, rat 56.1%; fox, owl, hen 2.3%


n= 8  home chosen: cat, dog, rat 80.0%; fox, owl, hen 0.2%
n=16  home chosen: cat, dog, rat 95.5%; fox, owl, hen 0.0%


A larger $n$ pushes harder toward whatever the reward model scores highest, and further from what the base model would usually say. The KL penalty is the dial that limits this. For a fixed set of responses, the policy that maximizes reward minus $\beta \operatorname{KL}$ has an exact form: $\pi^* \propto \pi_{\text{ref}} \, e^{R/\beta}$, the reference's probabilities reweighted by the exponentiated reward. The toy has only six responses per prompt, so the block computes it directly for four values of $\beta$, in log space so that tiny probabilities don't turn the KL into `nan`.

In [6]:
for beta in (10, 3, 1, 0.3):
    row = []
    for group in (trained, rest):
        batch = prompts(group)
        ref_logp, R = place_dist(base, batch).log(), rm_scores(rm, batch)   # log π_ref and R, 48 × 6 each
        logp = torch.log_softmax(ref_logp + R / beta, dim=-1)       # log π*, where π* ∝ π_ref e^(R/β)
        row += [logp.exp()[torch.arange(len(batch)), home_index(batch)].mean().item(), kl_to_ref(logp, ref_logp)]
    print(f"beta {beta:4}: cat, dog, rat {row[0]:.1%} (KL {row[1]:.2f}); fox, owl, hen {row[2]:.1%} (KL {row[3]:.2f})")

beta   10: cat, dog, rat 34.8% (KL 0.08); fox, owl, hen 13.7% (KL 0.04)


beta    3: cat, dog, rat 76.6% (KL 0.83); fox, owl, hen 5.8% (KL 0.52)
beta    1: cat, dog, rat 99.6% (KL 1.66); fox, owl, hen 0.1% (KL 1.68)


beta  0.3: cat, dog, rat 100.0% (KL 1.69); fox, owl, hen 0.0% (KL 1.80)


## Direct preference optimization

In words: raise the preferred response's probability relative to the starting model, lower the rejected one's, and stop caring once the gap is large. $\beta$ sets how large is enough. It's Bradley–Terry with $\beta \log(\pi_\theta/\pi_{\text{ref}})$ in place of the reward. What DPO removes is the reward model and the sampling loop, not the human comparisons: the run below still trains on 270 of them.

In [7]:
def dpo_loss(pol_pref, pol_rej, ref_pref, ref_rej, beta):   # sequence log-probabilities, one per pair
    margin = (pol_pref - ref_pref) - (pol_rej - ref_rej)     # how much more the policy favors y+ than the reference does
    return -F.logsigmoid(beta * margin).mean()

This time the pairs cover **all six** animals, but only some phrasings: the first three verbs with the first three prepositions, 9 of the 16 combinations (54 prompts, 270 pairs). The test uses the other 7 combinations, which DPO never trains on. Gentle settings: learning rate 0.0001, 100 steps, $\beta = 0.1$. `dpo` loads two copies of the base model, a policy to train and a frozen reference, and calls `dpo_loss` on each batch.

In [8]:
gentle = dpo(pairs_for(train_prompts), dpo_loss, beta=0.1, steps=100, lr=0.0001, seed=0)   # about 4 s
for name, model in (("base", base), ("gentle DPO", gentle)):
    print(f"{name:10}  home, unseen phrasings {home_rate(model, test_prompts):.1%}  validation loss {val_loss(model):.3f}")

base        home, unseen phrasings 17.7%  validation loss 0.244
gentle DPO  home, unseen phrasings 78.7%  validation loss 0.357


## When the score lies: over-optimization

Now the same DPO run with aggressive settings: learning rate 0.001 and 200 steps. By the preference measure, the home rate among the six places, it's perfect, or nearly. The block also splits each model's probability after an unseen prompt three ways: the preferred place, the other five places, and anything else. "Anything else" is 1 minus the six places' total, the chance that the next characters don't spell a place at all.

In [9]:
aggressive = dpo(pairs_for(train_prompts), dpo_loss, beta=0.1, steps=200, lr=0.001, seed=0)   # about 7 s
for name, model in (("base", base), ("gentle DPO", gentle), ("aggressive", aggressive)):
    preferred, other, rest_of = breakdown(model, test_prompts)
    print(f"{name:10}  home {home_rate(model, test_prompts):6.1%}  |  preferred {preferred:5.1%}  other places {other:5.1%}  anything else {rest_of:5.1%}")
print(f"aggressive validation loss {val_loss(aggressive):.3f}")
print(repr(generate(aggressive, "the fox lay under the", 12)))
print(repr(generate(aggressive, "the ", 60, seed=0)))

base        home  17.7%  |  preferred 17.6%  other places 82.0%  anything else  0.4%
gentle DPO  home  78.7%  |  preferred 47.6%  other places 17.2%  anything else 35.2%
aggressive  home 100.0%  |  preferred 30.2%  other places  0.0%  anything else 69.8%
aggressive validation loss 2.187
'the fox lay under therer ehehet a'
'the btd the de oty the het and the rd oy the he hen and the cat '


---

← [Guide 8: Fine-tuning](08-finetuning.ipynb)

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).